In [ ]:
"""
Load pretrained GPT-2 from HuggingFace, replace every self-attention
layer with STARLayerParallel, freeze everything except the new STAR
layers, and continue training on FineWeb-Edu (streaming) inside
Google Colab.

Only the STAR layers are trainable. Token/position embeddings, layer
norms, MLPs, and the (tied) lm_head all come from pretrained GPT-2
and are frozen.

Features:
- checkpoint saving (model, optimizer, step, loss history)
- live loss printing (configurable grouping of steps)
- resume training from a checkpoint
- gradient accumulation
"""

# %% ------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------
import math
import time

import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset
from tqdm.auto import tqdm
from transformers import GPT2LMHeadModel

# %% ------------------------------------------------------------------
# Config (edit these instead of using argparse)
# -----------------------------------------------------------------------
CONFIG = {
    # --- data ---
    "dataset_name": "HuggingFaceFW/fineweb-edu",
    "dataset_subset": "sample-10BT",
    "seq_len": 512,          # must be <= model's n_positions (1024 for base gpt2)
    "batch_size": 8,

    # --- model ---
    "model_name": "gpt2",    # "gpt2" (124M) / "gpt2-medium" / "gpt2-large" / "gpt2-xl"

    # --- training ---
    "max_steps": 20000,
    "lr": 5e-5,
    "min_lr_ratio": 0.1,
    "lr_decay_steps": 20000,
    "weight_decay": 0.1,
    "warmup_steps": 0,
    "grad_clip": 1.0,
    "gradient_accumulation_steps": 1,   # set >1 to accumulate gradients
    "eval_interval": 100,
    "eval_iters": 20,
    "print_interval": 6,                # group actual steps for printing
    "log_interval": 5,
    "checkpoint_interval": 500,         # save checkpoint every N optimizer steps

    # --- misc ---
    "seed": 1337,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "checkpoint_path": "/content/model.pt",
}

torch.manual_seed(CONFIG["seed"])
print(f"Using device: {CONFIG['device']}")
if CONFIG["device"] == "cpu":
    print("WARNING: no GPU detected. In Colab: Runtime > Change runtime type > GPU.")


# %% ------------------------------------------------------------------
# Tokenizer (GPT-2 BPE — vocab matches HF gpt2 exactly: 50257 tokens)
# -----------------------------------------------------------------------
tokenizer = tiktoken.get_encoding("gpt2")
VOCAB_SIZE = tokenizer.n_vocab


# %% ------------------------------------------------------------------
# Streaming dataset
# -----------------------------------------------------------------------
class FineWebEduStream(IterableDataset):
    def __init__(self, seq_len, split="train", subset="sample-10BT"):
        super().__init__()
        self.seq_len = seq_len
        self.split = split
        self.subset = subset

    def __iter__(self):
        ds = load_dataset(
            CONFIG["dataset_name"],
            name=self.subset,
            split=self.split,
            streaming=True,
        )
        ds = ds.shuffle(seed=CONFIG["seed"], buffer_size=10_000)

        buffer = []
        eot = tokenizer.eot_token

        for example in ds:
            text = example["text"]
            if not text:
                continue
            ids = tokenizer.encode_ordinary(text)
            ids.append(eot)
            buffer.extend(ids)

            while len(buffer) >= self.seq_len + 1:
                chunk = buffer[: self.seq_len + 1]
                buffer = buffer[self.seq_len + 1 :]
                x = torch.tensor(chunk[:-1], dtype=torch.long)
                y = torch.tensor(chunk[1:], dtype=torch.long)
                yield x, y


def make_loader(split, subset):
    dataset = FineWebEduStream(seq_len=CONFIG["seq_len"], split=split, subset=subset)
    return DataLoader(dataset, batch_size=CONFIG["batch_size"])


train_loader = make_loader("train", CONFIG["dataset_subset"])
val_loader = make_loader("train", CONFIG["dataset_subset"])


# %% ------------------------------------------------------------------
# STAR layer (replaces attention)
# -----------------------------------------------------------------------
class STARLayerParallel(nn.Module):
    """Causal token-mixing layer (replaces attention).

    add_residual=True  -> standalone use: returns x + transform(x)
    add_residual=False -> "attention-slot" use: returns only transform(x),
                           letting the surrounding block add its own
                           residual (this is how it's used inside GPT-2
                           blocks, since GPT2Block already does
                           `residual + attn_output` around ln_1).
    """
    def __init__(self, d_model, eps: float = 1e-6, add_residual: bool = True):
        super().__init__()
        self.add_residual = add_residual
        self.tok_proj = nn.Sequential(
            nn.Linear(d_model, d_model, bias=False), nn.SiLU(),
            nn.Linear(d_model, d_model, bias=False),
        )
        self.scene_proj = nn.Sequential(
            nn.Linear(d_model, d_model, bias=False), nn.SiLU(),
            nn.Linear(d_model, d_model, bias=False),
        )
        self.W_out = nn.Linear(d_model, d_model, bias=False)
        self.scene_norm = nn.LayerNorm(d_model)
        self.gate_proj = nn.Linear(d_model, d_model, bias=False)
        self.eps = eps

    def forward(self, x):
        B, T, D = x.shape
        raw = self.tok_proj(x)
        gate = torch.sigmoid(self.gate_proj(x))
        weighted = raw * gate

        cum_weighted = torch.cumsum(weighted, dim=1)
        cum_gate = torch.cumsum(gate, dim=1)

        cum_weighted_excl = torch.zeros_like(cum_weighted)
        cum_gate_excl = torch.zeros_like(cum_gate)
        cum_weighted_excl[:, 1:, :] = cum_weighted[:, :-1, :]
        cum_gate_excl[:, 1:, :] = cum_gate[:, :-1, :]

        scene_raw = cum_weighted_excl / (cum_gate_excl + self.eps)
        scene_before = self.scene_norm(scene_raw)

        delta = torch.tanh(raw * (1.0 + self.scene_proj(scene_before)))
        out = self.W_out(scene_before + delta)
        return x + out if self.add_residual else out


class STARAttentionWrapper(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.star = STARLayerParallel(d_model, add_residual=False)

    def forward(self, hidden_states, *args, **kwargs):
        attn_output = self.star(hidden_states)
        return attn_output, None


# %% ------------------------------------------------------------------
# Build model: pretrained GPT-2 with attention swapped for STAR,
# everything except the STAR layers frozen
# -----------------------------------------------------------------------
def build_star_gpt2(model_name: str):
    print(f"Loading pretrained '{model_name}'...")
    model = GPT2LMHeadModel.from_pretrained(model_name)
    model.config.use_cache = False  # STAR always recomputes the full sequence

    d_model = model.config.n_embd
    for block in model.transformer.h:
        block.attn = STARAttentionWrapper(d_model)

    # Freeze absolutely everything first...
    for p in model.parameters():
        p.requires_grad = False

    # ...then unfreeze only the newly-inserted STAR layers.
    for block in model.transformer.h:
        for p in block.attn.parameters():
            p.requires_grad = True

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params: {total/1e6:.2f}M | Trainable (STAR only): {trainable/1e6:.2f}M "
          f"({100*trainable/total:.1f}%)")
    return model


model = build_star_gpt2(CONFIG["model_name"])
model.to(CONFIG["device"])
assert CONFIG["seq_len"] <= model.config.n_positions, (
    f"seq_len ({CONFIG['seq_len']}) must be <= model n_positions ({model.config.n_positions})"
)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),  # STAR params only
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    betas=(0.9, 0.95),
)


def lr_at_step(step):
    if step < CONFIG["warmup_steps"]:
        return CONFIG["lr"] * (step + 1) / CONFIG["warmup_steps"]
    if step >= CONFIG["lr_decay_steps"]:
        return CONFIG["lr"] * CONFIG["min_lr_ratio"]
    progress = (step - CONFIG["warmup_steps"]) / max(1, CONFIG["lr_decay_steps"] - CONFIG["warmup_steps"])
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_ratio = CONFIG["min_lr_ratio"]
    return CONFIG["lr"] * (min_ratio + (1 - min_ratio) * coeff)


# %% ------------------------------------------------------------------
# Forward + loss helper (HF model returns .logits; targets are already
# shifted by one position in the dataset, so we do NOT use HF's built-in
# `labels=` shifting — cross-entropy is computed directly against y)
# -----------------------------------------------------------------------
def forward_loss(x, y):
    logits = model(input_ids=x).logits
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
    return logits, loss


# %% ------------------------------------------------------------------
# Validation loss
# -----------------------------------------------------------------------
@torch.no_grad()
def estimate_val_loss(val_iter):
    model.eval()
    losses = []
    for _ in range(CONFIG["eval_iters"]):
        x, y = next(val_iter)
        x, y = x.to(CONFIG["device"]), y.to(CONFIG["device"])
        _, loss = forward_loss(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


# %% ------------------------------------------------------------------
# Checkpoint helpers
# -----------------------------------------------------------------------
def save_checkpoint(step, train_steps, train_losses, val_steps, val_losses):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "train_steps": train_steps,
        "train_losses": train_losses,
        "val_steps": val_steps,
        "val_losses": val_losses,
        "config": CONFIG,
    }
    torch.save(checkpoint, CONFIG["checkpoint_path"])
    print(f"Checkpoint saved to {CONFIG['checkpoint_path']} at step {step}")


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=CONFIG["device"])
    model.load_state_dict(ckpt["model_state_dict"])
    #optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_step = ckpt["step"]
    train_steps = ckpt.get("train_steps", [])
    train_losses = ckpt.get("train_losses", [])
    val_steps = ckpt.get("val_steps", [])
    val_losses = ckpt.get("val_losses", [])
    print(f"Resumed from checkpoint at step {start_step}")
    return start_step, train_steps, train_losses, val_steps, val_losses


# %% ------------------------------------------------------------------
# Training loop (with gradient accumulation, resume support, and step printing)
# -----------------------------------------------------------------------
def train(resume_from=None):
    # Data iterators (re-created on resume, same seed ensures reproducibility)
    train_iter = iter(train_loader)
    val_iter = iter(val_loader)

    # Loss history
    if resume_from is not None:
        start_step, train_steps, train_losses, val_steps, val_losses = load_checkpoint(resume_from)
    else:
        start_step = 0
        train_steps, train_losses = [], []
        val_steps, val_losses = [], []

    acc_steps = CONFIG["gradient_accumulation_steps"]
    effective_batch_size = CONFIG["batch_size"] * acc_steps
    print_interval = CONFIG["print_interval"]

    model.train()
    pbar = tqdm(range(start_step, CONFIG["max_steps"]), desc="training")
    t0 = time.time() - (start_step * effective_batch_size * CONFIG["seq_len"] / 1e6)

    for step in pbar:
        # Adjust learning rate (per optimizer step)
        lr = lr_at_step(step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        # Gradient accumulation loop
        running_loss = 0.0
        for micro_step in range(acc_steps):
            x, y = next(train_iter)
            x, y = x.to(CONFIG["device"]), y.to(CONFIG["device"])

            _, loss = forward_loss(x, y)
            # Normalize loss to keep effective learning rate unchanged
            loss = loss / acc_steps
            loss.backward()
            running_loss += loss.item() * acc_steps  # unscaled loss for logging

        # Clip gradients and optimizer step (only STAR params have grads)
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), CONFIG["grad_clip"]
        )
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        # Logging (average loss over accumulation steps)
        train_steps.append(step)
        train_losses.append(running_loss / acc_steps)

        if step % CONFIG["log_interval"] == 0 or step == start_step:
            elapsed = max(1, time.time() - t0)
            total_tokens = (step + 1) * effective_batch_size * CONFIG["seq_len"]
            tokens_per_sec = total_tokens / elapsed
            pbar.set_postfix(
                loss=f"{running_loss/acc_steps:.3f}",
                lr=f"{lr:.2e}",
                tok_s=f"{tokens_per_sec:,.0f}"
            )

        # Print step info at configurable intervals (e.g., every 6 actual steps)
        if (step + 1) % print_interval == 0 or step == CONFIG["max_steps"] - 1:
            display_step = (step + 1) // print_interval
            print(f"step {display_step} loss:{train_losses[-1]:.3f} (actual step {step+1})")

        # Validation
        if step % CONFIG["eval_interval"] == 0 or step == CONFIG["max_steps"] - 1:
            v_loss = estimate_val_loss(val_iter)
            val_steps.append(step)
            val_losses.append(v_loss)
            print(f"Validation loss at step {step+1}: {v_loss:.3f}")

        # Save full checkpoint
        if step % CONFIG["checkpoint_interval"] == 0 and step > start_step:
            save_checkpoint(step, train_steps, train_losses, val_steps, val_losses)

    # Final save
    save_checkpoint(CONFIG["max_steps"], train_steps, train_losses, val_steps, val_losses)
    print(f"Training finished. Final model saved to {CONFIG['checkpoint_path']}")
    return train_steps, train_losses, val_steps, val_losses


# %% ------------------------------------------------------------------
# Sample generation (manual loop — STAR has no KV cache, so every step
# recomputes the forward pass over the current prefix)
# -----------------------------------------------------------------------
@torch.no_grad()
def generate(idx, max_new_tokens, temperature=1.0, top_k=50):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx if idx.size(1) <= CONFIG["seq_len"] else idx[:, -CONFIG["seq_len"]:]
        logits = model(input_ids=idx_cond).logits
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx


def sample(prompt="The history of ", max_new_tokens=100):
    ids = tokenizer.encode_ordinary(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=CONFIG["device"])
    out = generate(idx, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0].tolist())


# %% ------------------------------------------------------------------
# Entry point
# -----------------------------------------------------------------------
if __name__ == "__main__":
    # Example: resume from a previous checkpoint
    train(resume_from="/content/model.pt")
    #train()
    print("\n--- sample generation ---")
    print(sample())

Using device: cuda
Loading pretrained 'gpt2'...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total params: 138.58M | Trainable (STAR only): 42.49M (30.7%)
Resumed from checkpoint at step 1500


training:   0%|          | 0/18500 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Validation loss at step 1501: 5.050
step 251 loss:5.068 (actual step 1506)
step 252 loss:4.911 (actual step 1512)
step 253 loss:5.219 (actual step 1518)
step 254 loss:4.990 (actual step 1524)
step 255 loss:5.037 (actual step 1530)
step 256 loss:5.220 (actual step 1536)
step 257 loss:5.394 (actual step 1542)
step 258 loss:5.101 (actual step 1548)
step 259 loss:5.147 (actual step 1554)
step 260 loss:4.933 (actual step 1560)
step 261 loss:4.992 (actual step 1566)
step 262 loss:4.938 (actual step 1572)
step 263 loss:5.167 (actual step 1578)
step 264 loss:5.219 (actual step 1584)
step 265 loss:4.930 (actual step 1590)
step 266 loss:5.226 (actual step 1596)
Validation loss at step 1601: 4.957
step 267 loss:4.809 (actual step 1602)
step 268 loss:5.270 (actual step 1608)
step 269 loss:5.219 (actual step 1614)
step 270 loss:5.163 (actual step 1620)
step 271 loss:4.858 (actual step 1626)
step 272 loss:5.367 (actual step 1632)
step 273 loss:5.059 (actual step 1638)
step 274 loss:5.254 (actual ste

KeyboardInterrupt: 

In [ ]:
model = build_star_gpt2(CONFIG["model_name"])
model.to(CONFIG["device"])
load_checkpoint("/content/model.pt")
def sample(prompt="The history of ", max_new_tokens=100):
    ids = tokenizer.encode_ordinary(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=CONFIG["device"])
    out = generate(idx, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0].tolist())
while True:
  print(sample(prompt=input()))